In [2]:
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import psutil

In [3]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 15.6%


In [4]:
## NOTE: ADJUST HARD CODED MEANS IN NODE FEATURE MATRIX CELL (2-3 below this one)

In [5]:
# upload graph and verify intact

with open('../data/constructs/canwell_graphNov26.pkl', 'rb') as f:
    data = pickle.load(f)

G = data['G']
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

# check a random node
import random
node = random.choice(list(G.nodes()))
print(G.nodes[node])

Nodes: 714,421
Edges: 2,850,985
{'dem_idx': (567, 1228), 'elev': 1507.155029296875, 'slope': 26.334964752197266, 'aspect': 254.4375457763672, 'doubslope': 38.50175857543945, 'diff': -0.5445556640625, 'is_slope': True, 'is_basin': False}


In [6]:
# convert networkx graph to a pytorch data object

print("building node feature matrix...")

num_features = 4
nodes = list(G.nodes(data=True))
node_list = [n for n, _ in nodes]
node_to_idx = {n: i for i, n in enumerate(node_list)}

# build feature matrix: [elev, slope, aspect, doubslope]
# diff handled as the target
num_nodes = len(node_list)
x = torch.zeros((num_nodes, num_features), dtype=torch.float)
y = torch.zeros(num_nodes, dtype=torch.float)
is_slope = torch.zeros(num_nodes, dtype=torch.bool)
is_basin = torch.zeros(num_nodes, dtype=torch.bool)
has_diff = torch.zeros(num_nodes, dtype=torch.bool)


# hard code these for now
ELEV_MEAN = 1157.0
SLOPE_MEAN = 20.5
ASPECT_MEAN = 196.0 # most important
DOUBSLP_MEAN = 34.0


for i, (n,attr) in enumerate(nodes):
    x[i, 0] = attr['elev'] if attr['elev'] is not None else ELEV_MEAN
    x[i, 1] = attr['slope'] if attr['slope'] is not None else SLOPE_MEAN
    x[i, 2] = attr['aspect'] if attr['aspect'] is not None else ASPECT_MEAN
    x[i, 3] = attr['doubslope'] if attr['doubslope'] is not None else DOUBSLP_MEAN

    if attr['diff'] is not None:
        y[i] = attr['diff']
        has_diff[i] = True

    is_slope[i] = attr['is_slope']
    is_basin[i] = attr['is_basin']

print(f"feature matrix shape: {x.shape}")
print(f"nodes with diff values: {has_diff.sum():,}")
print(f"slope nodes: {is_slope.sum():,}")
print(f"basin nodes: {is_basin.sum():,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

building node feature matrix...
feature matrix shape: torch.Size([714421, 4])
nodes with diff values: 712,986
slope nodes: 387,315
basin nodes: 327,106
RAM: 27.9%


In [7]:
# How many nodes had None values that got filled with their feature mean
none_slope    = sum(1 for _, attr in nodes if attr['slope']     is None)
none_aspect   = sum(1 for _, attr in nodes if attr['aspect']    is None)
none_doubslop = sum(1 for _, attr in nodes if attr['doubslope'] is None)
none_elev     = sum(1 for _, attr in nodes if attr['elev']      is None)

print(f"None values filled with mean:")
print(f"  elev:      {none_elev:,}")
print(f"  slope:     {none_slope:,}")
print(f"  aspect:    {none_aspect:,}")
print(f"  doubslope: {none_doubslop:,}")
print(f"  diff missing: {(~has_diff).sum():,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

None values filled with mean:
  elev:      0
  slope:     0
  aspect:    0
  doubslope: 0
  diff missing: 1,435
RAM: 27.9%


In [8]:
torch.save({
    'x': x,
    'y': y,
    'is_slope': is_slope,
    'is_basin': is_basin,
    'has_diff': has_diff,
    'node_to_idx': node_to_idx
}, '../data/constructs/canwell_featuresNov26.pt')

print("saved")

saved


In [9]:
# after this, consider restarting the kernel
# run the imports and ram check from cells 1 and 2

In [5]:
print("loading features...")
feat = torch.load('../data/constructs/canwell_featuresNov26.pt')
node_to_idx = feat['node_to_idx']
print(f"ram after features load: {psutil.virtual_memory().percent:.1f}%")

loading features...
ram after features load: 16.2%


In [6]:
print("loading graph...")
with open('../data/constructs/canwell_graphNov26.pkl', 'rb') as f:
    data = pickle.load(f)
G = data['G']
print(f"ram after G load: {psutil.virtual_memory().percent:.1f}%")

loading graph...
ram after G load: 27.5%


In [7]:
del data
import gc
gc.collect()
print(f"ram after del data: {psutil.virtual_memory().percent:.1f}%")

ram after del data: 27.5%


In [8]:
print("building edge index...")
n = G.number_of_edges()
src = np.empty(n, dtype=np.int32)
dst = np.empty(n, dtype=np.int32)

for i, (u,v) in enumerate(G.edges()):
    if i%2000000 == 0:
        print(f" {i:,}/{n:,}, ram: {psutil.virtual_memory().percent:.1f}%")
    src[i] = node_to_idx[u]
    dst[i] = node_to_idx[v]

del G
import gc
gc.collect()
print(f"ram after del G: {psutil.virtual_memory().percent:.1f}%")

building edge index...
 0/2,850,985, ram: 27.5%
 2,000,000/2,850,985, ram: 27.8%
ram after del G: 16.7%


In [9]:
# build undirected edge index
edge_index = torch.tensor(
    np.vstack([np.concatenate([src,dst]),
               np.concatenate([dst,src])]),
    dtype=torch.long
)

del src, dst
gc.collect()

print(f"edge index shape: {edge_index.shape}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

edge index shape: torch.Size([2, 5701970])
ram: 16.8%


In [11]:
torch.save(edge_index, '../data/constructs/canwell_edgidxNov26.pt')
print("saved")

saved
